# BaseModel vs SFT — Colab GPU

Выбери Runtime → Change runtime type → T4 GPU, затем Run all.

По умолчанию: повторяемость на 16 примерах, полный base zero-shot, полный SFT zero-shot.
Обучение SFT включается RUN_TRAINING=True; готовый адаптер обучен отдельно на Kaggle.

Код берётся из GitHub, из внешнего пакета импортируются только данные и адаптер. Результаты сохраняются в Drive либо скачиваемый ZIP. Без GPU ноутбук остановится. См. docs/evaluation.md.

In [ ]:
import torch, subprocess, sys, os, json, time, hashlib, zipfile, shutil
from pathlib import Path
assert torch.cuda.is_available(), 'Нужен GPU: Runtime → Change runtime type → T4 GPU, затем Run all.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


## Настройки

In [ ]:
MODEL = 'Qwen/Qwen2.5-0.5B'
MODE = 'zero'
RUN_TRAINING = False
CHECK_REPRODUCIBILITY = True
EVALUATE_SFT = True
BUNDLE_SHA256 = '2efb3095ae5541c180fcbfcf6691338699036bb8da133b6cc314b68842eb84c2'
REPO_URL = 'https://github.com/Pavel-V418/BaseModel-vs-SFT.git'
REPO_REF = 'participant-2/evaluation'  # Для точного повторения укажи SHA коммита.
TEST_SHA256 = '07def41c02ff1bef775a2dd7ca0fc676b405d38b8e762d9bb32624e9aa328b03'


## Данные и хранилище
Сначала используется архив, загруженный через Files → Upload to session storage. Если его нет, ноутбук подключает Drive и собирает две части.
При недоступном Drive результаты сохраняются в Colab; последняя ячейка скачивает ZIP.

In [ ]:
local_bundle = Path('/content/basemodel_colab_bundle.zip')
PROJECT_DRIVE = Path('/content/drive/MyDrive/BaseModel-vs-SFT-Colab')
if local_bundle.is_file():
    print('Используем прямую загрузку в Colab; подключение Drive не требуется.', flush=True)
    deadline = time.monotonic() + 600
    while local_bundle.stat().st_size < 116789255 and time.monotonic() < deadline:
        print(f'Загрузка архива: {local_bundle.stat().st_size / 1e6:.1f}/116.8 MB', flush=True)
        time.sleep(5)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    parts = [PROJECT_DRIVE / f'basemodel_colab_bundle.zip.part{i:03d}' for i in (1,2)]
    assert all(p.is_file() for p in parts), 'Нет частей архива в папке проекта Drive'
    with local_bundle.open('wb') as out:
        for part in parts:
            with part.open('rb') as src:
                shutil.copyfileobj(src, out)
assert hashlib.sha256(local_bundle.read_bytes()).hexdigest() == BUNDLE_SHA256, 'Архив загружен не полностью или повреждён'
ROOT = Path('/content/BaseModel-vs-SFT-audited')
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
assert (ROOT/'.git').is_dir(), f'{ROOT} должна быть клоном репозитория'
origin = subprocess.check_output(['git', '-C', str(ROOT), 'remote', 'get-url', 'origin'], text=True).strip()
assert origin == REPO_URL, f'Неожиданный origin: {origin}'
subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
with zipfile.ZipFile(local_bundle) as z:
    for member in z.infolist():
        path = Path(member.filename)
        assert not path.is_absolute() and '..' not in path.parts, member.filename
        if path.parts[:2] not in [('BaseModel-vs-SFT', 'data'), ('BaseModel-vs-SFT', 'outputs')]:
            continue
        target = ROOT.joinpath(*path.parts[1:])
        assert target.resolve().is_relative_to(ROOT.resolve()), member.filename
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(member) as source, target.open('wb') as dest:
                shutil.copyfileobj(source, dest)
assert hashlib.sha256((ROOT/'data/test.jsonl').read_bytes()).hexdigest() == TEST_SHA256, 'Изменился test'
print('Код:', SOURCE_COMMIT)
os.chdir(ROOT)
DRIVE_RESULTS = Path('/content/drive/MyDrive').is_dir()
RESULTS_BASE = PROJECT_DRIVE if DRIVE_RESULTS else Path('/content/BaseModel-vs-SFT-Colab-results')
SESSION = RESULTS_BASE / 'runs' / time.strftime('%Y%m%d_%H%M%S')
SESSION.mkdir(parents=True, exist_ok=False)
ADAPTER = ROOT / 'outputs/sft_0.5b_nall_r16_lr1e-4_s42/final'
print('Архив проверен. Данные и адаптер распакованы.')
print('Результаты:', SESSION)
if not DRIVE_RESULTS:
    print('Результаты находятся на диске Colab: финальная ячейка скачает ZIP. До скачивания не завершай среду.')


## Библиотеки и проверки
CUDA-версия torch из Colab сохраняется. Настройки и библиотеки для base и SFT одинаковые.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_parse.py', 'tests/test_evaluate.py'], check=True)
records = [json.loads(line) for line in (ROOT/'data/test.jsonl').read_text().splitlines() if line.strip()]
assert len(records) == 1034, f'Ожидалось 1034 test, получено {len(records)}'
from collections import Counter
print('Test:', len(records), dict(Counter(r['difficulty'] for r in records)))
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(SESSION/'requirements-frozen.txt').write_text(freeze)
(SESSION/'environment.json').write_text(json.dumps({'model':MODEL, 'mode':MODE, 'gpu':torch.cuda.get_device_name(0), 'torch':torch.__version__, 'test_sha256':hashlib.sha256((ROOT/'data/test.jsonl').read_bytes()).hexdigest(), 'source_bundle_sha256':BUNDLE_SHA256, 'source_commit':SOURCE_COMMIT}, indent=2))


In [ ]:
def run_command(args):
    log_name = 'train.log' if 'src.train' in args else args[args.index('--run_name')+1]+'.log'
    print('Запуск:', ' '.join(map(str,args)), flush=True)
    with (SESSION/log_name).open('w') as f:
        with subprocess.Popen(list(map(str,args)), cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ,'PYTHONUNBUFFERED':'1'}) as proc:
            try:
                for line in proc.stdout:
                    print(line, end='', flush=True)
                    f.write(line)
                    f.flush()
                rc = proc.wait()
            except BaseException:
                proc.terminate()
                proc.wait()
                raise
    assert rc == 0, f'Ошибка {rc}; лог: {SESSION/log_name}'

def evaluate(run_name, adapter=None, limit=None):
    args = [sys.executable, '-u', '-m', 'src.evaluate', '--model', MODEL, '--mode', MODE, '--run_name', run_name, '--results_dir', SESSION]
    if adapter is not None:
        args += ['--adapter', str(adapter)]
    if limit is not None:
        args += ['--limit', str(limit)]
    run_command(args)
    return json.loads((SESSION/f'{run_name}_eval_metrics.json').read_text())


## Необязательно: новое обучение SFT
Только при RUN_TRAINING=True. Новый адаптер и чекпоинты сохраняются в отдельную папку Drive, затем новый адаптер используется в сравнении.

In [ ]:
if RUN_TRAINING:
    train_name = 'colab_sft_0.5b_r16_' + SESSION.name
    train_output = SESSION / 'trained' / train_name
    train_output.mkdir(parents=True, exist_ok=True)
    (ROOT/'outputs'/train_name).symlink_to(train_output, target_is_directory=True)
    run_command([sys.executable, '-u', '-m', 'src.train', '--config', 'configs/sft_0.5b.yaml', '--set', f'run_name={train_name}', '--set', 'report_to=none'])
    for f in (ROOT/'results').glob(f'{train_name}*'):
        shutil.copy2(f, SESSION/f.name)
    ADAPTER = train_output/'final'
    assert (ADAPTER/'adapter_model.safetensors').is_file()
else:
    print('Готовый адаптер:', ADAPTER)


## Проверка base, адаптера и повторяемости
Два запуска на одинаковых 16 примерах должны дать одинаковые ответы и метрики.

In [ ]:
def check_pair(prefix, adapter=None):
    a = evaluate(prefix+'_a', adapter=adapter, limit=16)
    if CHECK_REPRODUCIBILITY:
        b = evaluate(prefix+'_b', adapter=adapter, limit=16)
        assert a['overall'] == b['overall'] and a['by_difficulty'] == b['by_difficulty'], 'Метрики различаются'
        assert (SESSION/f'{prefix}_a_eval.jsonl').read_text() == (SESSION/f'{prefix}_b_eval.jsonl').read_text(), 'Ответы различаются'
        print('Повторяемость подтверждена:', prefix)
    return a
smoke_base = check_pair('smoke_base')
if EVALUATE_SFT:
    assert (ADAPTER/'adapter_config.json').is_file(), 'Не найден адаптер'
    smoke_sft = check_pair('smoke_sft', ADAPTER)


## Полный base zero-shot
1034 примера, batch=16, greedy decoding, max_new_tokens=256, left padding.

In [ ]:
base_metrics = evaluate('base_0.5b_'+MODE)
print(json.dumps(base_metrics, ensure_ascii=False, indent=2))


## Полный SFT с теми же настройками

In [ ]:
if EVALUATE_SFT:
    sft_metrics = evaluate('sft_0.5b_'+MODE, adapter=ADAPTER)
    print(json.dumps(sft_metrics, ensure_ascii=False, indent=2))


## Сравнение и архив результатов

In [ ]:
import pandas as pd
comparison = {'base':base_metrics['overall']}
if EVALUATE_SFT:
    comparison['sft'] = sft_metrics['overall']
display(pd.DataFrame(comparison).T)
(SESSION/'comparison.json').write_text(json.dumps(comparison, ensure_ascii=False, indent=2))
archive = shutil.make_archive(str(SESSION.parent/(SESSION.name+'_results')), 'zip', SESSION)
print('Результаты:', SESSION)
print('Архив:', archive)

if not DRIVE_RESULTS:
    from google.colab import files
    files.download(archive)
